In [ ]:
from pathlib import Path
import sys

def _find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    return start

repo_root = _find_repo_root(Path.cwd())
workspace_root = repo_root.parent
candidate_src_dirs = [
    repo_root / "src",
    workspace_root / "abstractgraph" / "src",
    workspace_root / "abstractgraph-ml" / "src",
    workspace_root / "abstractgraph-generative" / "src",
]
for src_dir in candidate_src_dirs:
    if src_dir.exists():
        src_str = str(src_dir)
        if src_str not in sys.path:
            sys.path.insert(0, src_str)


# Interpolate Graph Demo
Load PubChem assay graphs and render a small sample.


In [ ]:
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
import warnings


In [ ]:
from coco_grape.data_loader.mol.mol_loader import PubChemLoader
from coco_grape.data_loader.loader import SupervisedDataSetLoader

assay_ids = ['2631','624249','651741','588350','463230','492952','743219','492992','463213']
assay_id = assay_ids[1]
#assay_id = '488975' #active 2,634
size = 3000
print(f"size: {size}")
use_equalized = True

def pubchem_loader():
    return PubChemLoader().load(assay_id)

graphs, targets = SupervisedDataSetLoader(
    pubchem_loader,
    size=size,
    use_equalized=use_equalized,
).load()
targets = np.array(targets)

from abstractgraph.utils import plot_graph_label_counts
_ = plot_graph_label_counts(graphs, top=20, title='Dataset info', log_scale=True)


In [ ]:

from abstractgraph.operators import *
from abstractgraph_ml.feasibility import FeasibilityEstimatorFeatureCannotExist, FeasibilityEstimator, FeasibilityEstimatorNumberOfNodesInRange
from abstractgraph_generative.interpolate import InterpolationEstimator
from abstractgraph_generative.interpolation import InterpolationGenerator
from abstractgraph_ml.estimators import GraphEstimator
from sklearn.ensemble import RandomForestClassifier
from abstractgraph.vectorize import AbstractGraphTransformer

nbits=14

decomposition_function = neighborhood(radius=(1,2))
graph_transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=decomposition_function,
    return_dense=True,
)

df = compose(neighborhood(radius=2), unlabel())
fe1 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)

df = add(neighborhood(radius=1), cycle())
fe2 = FeasibilityEstimatorFeatureCannotExist(decomposition_function=df, nbits=19)

min_size, max_size = np.quantile([len(g) for g in train_graphs], [0.25, 0.75])
fe3 = FeasibilityEstimatorNumberOfNodesInRange(min_size=min_size, max_size=max_size)

feasibility_estimators = [fe1, fe2, fe3]
feasibility_estimator = FeasibilityEstimator(feasibility_estimators)
feasibility_estimator.fit(train_graphs)

interpolation_estimator = InterpolationEstimator(
    graph_transformer=graph_transformer,
    n_iterations=4,
    n_samples=10,
    k=5,
    feasibility_estimator=feasibility_estimator,
    degree_penalty="auto",
    degree_penalty_mode="multiplicative",
    cut_radius=0,
    cut_scope="both",
)
interpolation_estimator.fit(train_graphs)

In [ ]:
from sklearn.model_selection import train_test_split
from abstractgraph_ml.estimators import GraphEstimator
from abstractgraph.vectorize import AbstractGraphTransformer

# Keep bioactivity labels (if any) for reference
bio_targets = targets

# Fit unsupervised outlier/inlier model (0=outlier, 1=inlier) on full dataset
vec = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=decomposition_function,
    return_dense=True,
    n_jobs=-1,
)
ge = GraphEstimator(transformer=vec).fit(graphs, targets=None)

# Use inlier probability and a user-defined threshold to create 0/1 labels
#  - 0 = outlier, 1 = inlier
oi_threshold = 0.5  # change to tune strictness
probs = ge.predict_proba(graphs, log=False)
classes = getattr(getattr(ge, 'estimator_', None), 'classes_', None)
if classes is None:
    inlier_col = 1 if probs.shape[1] > 1 else 0
else:
    try:
        inlier_col = int(np.where(classes == 1)[0][0])
    except Exception:
        # Fallback: if classes are not [0,1], choose the max class id as inlier
        inlier_col = int(np.argmax(classes))

p_inlier = probs[:, inlier_col]
oi_targets = (p_inlier >= float(oi_threshold)).astype(int)

# Split into test and train/reference using 0/1 semantics
test_size = 0.33
all_train_graphs, test_graphs, oi_all_train_targets, test_targets = train_test_split(
    graphs,
    oi_targets,
    test_size=test_size,
    stratify=oi_targets if len(np.unique(oi_targets)) > 1 else None,
    random_state=0,
)

reference_split_size = 0.5
train_graphs, reference_graphs, train_targets, reference_targets = train_test_split(
    all_train_graphs,
    oi_all_train_targets,
    test_size=reference_split_size,
    stratify=oi_all_train_targets if len(np.unique(oi_all_train_targets)) > 1 else None,
    random_state=0,
)

print(f"AID {assay_id} graphs: {len(graphs)} (train={len(train_graphs)}, reference={len(reference_graphs)}, test={len(test_graphs)})")
print(f"Inlier threshold: {oi_threshold:.2f}; inlier ratio (full set): {np.mean(oi_targets==1):.3f}; classes={classes}")



In [ ]:
from abstractgraph_generative.rewrite import rewrite
source = train_graphs[1]
donors = train_graphs[2:3]
samples = rewrite(source=source, donors=donors, decomposition_function=decomposition_function, nbits=nbits, n_samples=10, single_replacement=False, cut_radius=0, cut_scope="both", cut_include_edge_label=True)
print(f'#samples:{len(samples)}')
from abstractgraph.hashing import GraphHashDeduper
samples = GraphHashDeduper().fit_filter(samples)
samples = GraphHashDeduper().fit([source]+donors).filter(samples)
print(f'#unique samples:{len(samples)}')
#from abstractgraph.display import display_graphs
from coco_grape.visualizer.mol_display import draw_molecules as display_graphs
_ = display_graphs([source]+donors)
_ = display_graphs(samples[:7])
filtered_samples = feasibility_estimator.filter(samples)
print(f'#filtered:{len(filtered_samples)}')
_ = display_graphs(filtered_samples[:7*2])

In [ ]:
import random
source_idx=random.randrange(len(train_graphs))
destination_idx=random.randrange(len(train_graphs))
source,destination = train_graphs[source_idx], train_graphs[destination_idx]
_ = display_graphs([source,destination])
samples = interpolation_estimator.interpolate_idxs(source_idx, destination_idx)
print(f'#generated:{len(samples)}')
samples = GraphHashDeduper().fit_filter(samples)
samples = GraphHashDeduper().fit([source, destination]).filter(samples)
print(f'#accepted:{len(samples)}')
if samples:
    _ = display_graphs([source] + samples + [destination])
else:
    print('No novel results')

---

In [ ]:
generator = InterpolationGenerator(interpolation_estimator).fit(train_graphs)

#from abstractgraph.display import display_graphs
from coco_grape.visualizer.mol_display import draw_molecules as display_graphs
generated_graphs, generated_targets = generator.generate(
    n_rounds=3,
    n_pairs=15,
    n_iterations=3,
    best_of=2,
    prob_for_acceptance_interval=(0.55, 0.95),
    verbose=True,
    draw_func=display_graphs,
)

In [ ]:
def estimated_generative_quality(generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets):
    from NSPPK.nsppk import NSPPK
    vectorizer = NSPPK(radius=1, distance=4, connector=1, nbits=14, parallel=True)

    from sklearn.ensemble import ExtraTreesClassifier
    classifier = ExtraTreesClassifier(n_estimators=300, n_jobs=-1, random_state=42)

    from abstractgraph_generative.generative_performance import compute_expected_gain_weighted_equivalent_data_size
    results = compute_expected_gain_weighted_equivalent_data_size(
        generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets,
        vectorizer=vectorizer, classifier=classifier,
        fracional_size=(1,2,3,4,5,7,10,15),
        n_repeats=100,
    )
    print(f'Expected gain weighted equivalent data size:{results["expected_gain_weighted_equivalent_data_size"]:.3f}')

    from abstractgraph_generative.generative_performance import plot_expected_gain_weighted_equivalent_data_size
    _ = plot_expected_gain_weighted_equivalent_data_size(results)

In [ ]:
estimated_generative_quality(generated_graphs, generated_targets, train_graphs, train_targets, reference_graphs, reference_targets, test_graphs, test_targets)